# RobotMath2030 — Flow Matching vs Diffusion (Ch.08)

Same multimodal task; velocity field vs denoising.

Full chapter: [08_flow_matching_vs_diffusion](../chapters/08_flow_matching_vs_diffusion/)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    !git clone -q https://github.com/rsasaki0109/RobotMath2030.git 2>/dev/null || true
    %cd RobotMath2030
except ImportError:
    root = Path.cwd()
    if not (root / 'robotmath').exists() and (root.parent / 'robotmath').exists():
        %cd ..

!pip install -q -e ".[torch]"
%matplotlib inline

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from miniworlds.two_path_world import START, GOAL, collision_rate, generate_demonstrations
from robotmath.diffusion import DiffusionPolicy2D, FlowMatchingPolicy2D, FlowTrainConfig, TrainConfig
from robotmath.viz.plot_trajectory import draw_trajectories, draw_two_path_world

horizon = 24
demos, cond = generate_demonstrations(n_per_mode=48, horizon=horizon, seed=0)
test_cond = np.concatenate([START, GOAL])[None, :]
n_samples = 20

diffusion = DiffusionPolicy2D(TrainConfig(horizon=horizon, timesteps=20, epochs=80, seed=0))
diff_losses = diffusion.fit(demos, cond, verbose=False)
flow = FlowMatchingPolicy2D(FlowTrainConfig(horizon=horizon, epochs=120, sample_steps=20, seed=0))
flow_losses = flow.fit(demos, cond, verbose=False)

t0 = time.perf_counter()
diff_samples = diffusion.sample(test_cond, n_samples=n_samples)
diff_time = time.perf_counter() - t0
t0 = time.perf_counter()
flow_samples = flow.sample(test_cond, n_samples=n_samples, steps=20, integrator='heun')
flow_time = time.perf_counter() - t0

print(f'Diffusion collision: {collision_rate(diff_samples):.0%}  sample time: {diff_time:.2f}s')
print(f'Flow match collision: {collision_rate(flow_samples):.0%}  sample time: {flow_time:.2f}s')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
draw_two_path_world(axes[0])
draw_trajectories(axes[0], diff_samples, color='C0', alpha=0.55)
axes[0].set_title('Diffusion samples')
draw_two_path_world(axes[1])
draw_trajectories(axes[1], flow_samples, color='C2', alpha=0.55)
axes[1].set_title('Flow matching samples')
axes[2].plot(diff_losses, label='diffusion', color='C0')
axes[2].plot(flow_losses, label='flow', color='C2')
axes[2].set_xlabel('epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_title('Training loss')
fig.tight_layout()
plt.show()